In [1]:
import torch

- $Tensor.scatter_(dim, index, src, *, reduce=None)$
    - Writes values from src to self based on index tensor and dim. 
    - For dimension == dim, index values are selected.
    - For dimension != dim, src values in src are selected. 
    

- It is useful for one hot encoding from (Batch_size, 1) into (Batch_size, N)

In [12]:
labels = torch.tensor([
    [2], [0], [1], [3]])

print(labels.shape) # (Batch_size, 1)

one_hot = torch.zeros(4, 4)

one_hot.scatter_(dim=1, index=labels, value=1)

print(one_hot)


torch.Size([4, 1])
tensor([[0., 0., 1., 0.],
        [1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 0., 1.]])


- Convert multiple selected indices into the multi class label form

In [17]:
labels = torch.tensor([
    [2, 0], [0, 1], [1, 1], [3, 3]])

print(labels.shape) # (Batch_size, 2)

one_hot = torch.zeros(4, 4)

one_hot.scatter_(dim=1, index=labels, value=1)

print(one_hot)


torch.Size([4, 2])
tensor([[1., 0., 1., 0.],
        [1., 1., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 0., 1.]])


- TopK Masking

In [22]:
logits = torch.tensor([[1.2, 5.0, 0.1, 4.2, 0.5],
                       [0.2, 1.1, 6.3, 0.5, 3.1]])

_, topk = torch.topk(logits, k=2, dim=1)

print(topk)

mask = torch.zeros_like(logits)
mask.scatter_(dim=-1, index=topk, value=1)

print(mask)

print(logits * mask)

tensor([[1, 3],
        [2, 4]])
tensor([[0., 1., 0., 1., 0.],
        [0., 0., 1., 0., 1.]])
tensor([[0.0000, 5.0000, 0.0000, 4.2000, 0.0000],
        [0.0000, 0.0000, 6.3000, 0.0000, 3.1000]])


## torch.gather
- Gather should be used to gather things from different places. Gather should be a reducer. 
- Transform N to K / extract chosen values

In [ ]:
q_values = torch.tensor([[0.5, 2.1, 0.1], [1.1, 0.2, 4.5]])

print(q_values)

# agent took action 1 in env1 and action 2 in second env 
actions_taken = torch.tensor([[1], [2]])

## gather values associated with selected indices
chosen_q_values = q_values.gather(dim=1, index=actions_taken)

print(chosen_q_values)

tensor([[0.5000, 2.1000, 0.1000],
        [1.1000, 0.2000, 4.5000]])
tensor([[2.1000],
        [4.5000]])


- Applying sorts to aligned data

In [3]:
scores = torch.tensor([[0.1, 0.9, 0.4]])
class_ids = torch.tensor([[10, 11, 12]])

_, sorted_idx = torch.sort(scores, descending=True)

sorted_classes = class_ids.gather(dim=1, index=sorted_idx)

print(sorted_classes)

tensor([[11, 12, 10]])


- Moe picking expert

In [5]:
# 1. The Router calculates probabilities for 4 tokens across 3 Experts
# Shape: (Batch=4, Experts=3)
router_probs = torch.tensor([
    [0.1, 0.2, 0.7],  # Token 0 prefers Expert 2
    [0.6, 0.3, 0.1],  # Token 1 prefers Expert 0
    [0.2, 0.5, 0.3],  # Token 2 prefers Expert 1
    [0.0, 0.9, 0.1]   # Token 3 prefers Expert 1
])

chosen_probs2, chosen_expert_idx = torch.topk(router_probs, k=1, dim=1)


chosen_probs = router_probs.gather(dim=1, index=chosen_expert_idx)

print(chosen_probs)

print(chosen_probs2)

tensor([[0.7000],
        [0.6000],
        [0.5000],
        [0.9000]])
tensor([[0.7000],
        [0.6000],
        [0.5000],
        [0.9000]])


- Why bother with gather

In [8]:
# 1. We have our clean, true probabilities
clean_probs = torch.tensor([
    [0.4, 0.5, 0.1],  
    [0.2, 0.2, 0.6]
])

# 2. We add random noise to encourage the model to try new experts
noisy_probs = clean_probs + torch.tensor([
    [0.3, 0.0, 0.0],  # Noise pushes Expert 0 into the lead!
    [0.0, 0.0, 0.1]
])

print(clean_probs)
print(noisy_probs)

_, chosen_expert_idx = torch.topk(noisy_probs, k=1, dim=1)

# select top values from the clean probs 

true_weights = clean_probs.gather(dim=1, index=chosen_expert_idx)

print(true_weights)

tensor([[0.4000, 0.5000, 0.1000],
        [0.2000, 0.2000, 0.6000]])
tensor([[0.7000, 0.5000, 0.1000],
        [0.2000, 0.2000, 0.7000]])
tensor([[0.4000],
        [0.6000]])
